In [ ]:
import random
from datasets import load_dataset, get_dataset_config_names

# =============================================================================
# 🤖 튜터의 인사말: 안녕하세요, 코딩 탐험가님! ✨
#
# 오늘의 목적지는 '대화형 AI 지능'의 심장부입니다!
# 우리가 다룰 데이터셋은 'heegyu/open-korean-instructions-v20231020'입니다.
# 이 데이터는 방대한 양의 한국어 질문-답변(Instructions) 대화 로그로 구성되어 있어요.
# 쉽게 말해, 수많은 사람들이 AI에게 "이거 어떻게 해줘?", "이거 알려줘"라고 물어보고,
# AI가 어떻게 답했는지를 정리한 'AI 학습 교과서' 같은 자료랍니다!
#
# 목표: 이 대화 로그들을 분석해서, 어떤 주제의 질문이 가장 많고,
# 우리가 나만의 '최애 챗봇' 프롬프트를 어떻게 만들지 시뮬레이션 해보는 거예요!
# 자, 그럼 코딩 마법을 시작해 볼까요? 파이팅! 💪
# =============================================================================

# --- 상수 설정 및 전역 변수 ---
DATASET_NAME = "heegyu/open-korean-instructions-v20231020"
SAMPLE_COUNT = 50  # 너무 많은 데이터는 느리니, 일단 50개의 샘플로 신나는 탐색을 해봐요!

# 1. 사용 가능한 Config 확인
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    
    # 기본 config를 선택합니다.
    selected_config = configs[0]
except Exception as e:
    print("ℹ️ 해당 데이터셋은 별도의 Config가 없거나 기본(default) 설정만 제공됩니다.")
    selected_config = None

# 2. 데이터셋 로딩 (스트리밍 우선 전략)
dataset = None
sample_data_iterator = None

print("\n=====================================================")
print("📡 STEP 1: 데이터셋 다운로드 및 샘플 추출 준비 (Streaming Mode 시도)")
print("=====================================================")

try:
    # 🚀 스트리밍 모드를 사용해 빠르게 데이터에 접근해봅시다! (성능 최적화!)
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✨ 성공! 스트리밍 모드(streaming=True)로 데이터셋을 로드했습니다. 매우 빠르죠? 🚀")
except Exception as e:
    print(f"⚠️ 경고! 스트리밍 모드 로드 실패 ({e.__class__.__name__} 에러가 발생할 수 있어요).")
    print("🔄 일반 모드로 전환하여 로드하겠습니다. (느릴 수 있어요)")
    # 스트리밍 실패 시, 일반 모드로 로드합니다.
    try:
        dataset = load_dataset(DATASET_NAME, split='train', streaming=False)
        print("✅ 성공! 일반 Dataset 모드로 데이터를 로드했습니다.")
    except Exception as e_fallback:
        print(f"🚨 치명적인 오류: 데이터셋 로드 자체에 실패했습니다. {e_fallback}")
        exit()

# 3. 샘플러블 데이터셋 확보 (가장 중요!)
print("\n=====================================================")
print(f"💡 STEP 2: {SAMPLE_COUNT}개의 샘플 데이터를 추출합니다...")
print("=====================================================")

# 스트리밍 데이터셋인지 확인하여 샘플링 패턴 적용 (핵심 로직!)
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    print("🌐 (탐지) 스트리밍 데이터셋입니다. .take() 메서드를 사용합니다.")
    # take()를 사용하여 상위 K개만 메모리로 불러옵니다.
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
    # 이 반복자(iterator)를 리스트로 변환하여 안전하게 순회합니다.
    sample_data_list = list(sampled_dataset_iterator)
else:
    # 일반 데이터셋 (Dataset)
    print("📚 (탐지) 일반 Dataset입니다. .take() 대신 리스트로 변환합니다.")
    # 리스트로 변환 후 슬라이싱을 통해 샘플링합니다.
    sample_data_list = list(dataset.select(range(min(SAMPLE_COUNT, 100)))).take(SAMPLE_COUNT)
    sample_data_list = list(sample_data_list)


# 4. 데이터 분석 및 창의적 활용 실습
print("\n=====================================================")
print(f"🧠 STEP 3: 데이터 분석 및 나만의 '대화 로그' 생성 실습 ({len(sample_data_list)}개 샘플)")
print("=====================================================")

total_turns = 0
user_turns_count = 0
assistant_turns_count = 0
topic_counts = {}

print("\n[📝 분석 목표]: 데이터셋에 등장하는 질문(User)과 답변(Assistant)의 패턴을 분석하여, 가장 인기 있는 질문 주제를 탐색합니다.")

# 샘플 리스트를 순회하며 데이터 분석을 진행합니다.
for i, sample in enumerate(sample_data_list):
    if not sample['conversations']:
        continue # 대화 기록이 없으면 건너뜁니다.

    # 대화는 리스트 형태로 [Role, Value] 쌍으로 되어 있습니다.
    for turn in sample['conversations']:
        # turn의 구조는 {'from': 'role', 'value': 'content'} 입니다.
        role = turn.get('from', 'UNKNOWN').lower()
        value = turn.get('value', '').strip()
        
        if value:
            total_turns += 1
            if role == 'human': # Human은 사용자(User) 질문입니다.
                user_turns_count += 1
                # 간단한 토픽 분석: 질문의 시작 몇 글자를 추출하여 카운트합니다.
                if len(value) >= 10:
                    topic = value[:10]
                    topic_counts[topic] = topic_counts.get(topic, 0) + 1
                else:
                    topic_counts[value] = topic_counts.get(value, 0) + 1
            elif role == 'gpt': # GPT는 AI의 답변(Assistant)입니다.
                assistant_turns_count += 1
                
# 5. 분석 결과 출력 및 시각적 피드백 (Print)
print("\n" + "=" * 70)
print("✨ 🌟 분석 결과 요약 🌟 ✨")
print("=" * 70)

# 📊 정량적 분석 결과 출력
print(f"총 분석된 샘플 수: {len(sample_data_list)}개")
print(f"총 대화 턴(Total Turns) 수: {total_turns}개")
print(f"👤 사용자 질문 (Human Role) 횟수: {user_turns_count}회")
print(f"🤖 AI 답변 (GPT Role) 횟수: {assistant_turns_count}회")

# 💡 트렌드 분석 (가장 많이 등장한 질문 주제 Top 3)
print("\n\n--- 🏆 Top 3 인기 질문 주제 (Top Query Trends) ---")
# 딕셔너리를 리스트로 변환 후, 값(횟수)을 기준으로 내림차순 정렬
sorted_topics = sorted(topic_counts.items(), key=lambda item: item[1], reverse=True)

for rank, (topic, count) in enumerate(sorted_topics[:3]):
    print(f"  {rank+1}위: '{topic[:20]}...' (등장 횟수: {count}회)")

# 🔬 창의적 활용 예시: 챗봇 프롬프트 생성 시뮬레이션
print("\n" + "=" * 70)
print("✨ 🎯 창의적 활용 예시: 나만의 '최애 챗봇' 프롬프트 설계 🎯 ✨")
print("=" * 70)

# 가장 많이 나온 주제를 가져와서 가상의 시나리오를 작성합니다.
if sorted_topics:
    best_topic, _ = sorted_topics[0]
    print(f"💖 **[분석 기반 시나리오]** 가장 많이 언급된 주제는 '{best_topic}' 관련 질문입니다.")
    print("-> 이 주제를 중심으로 나만의 챗봇(튜터)을 만들 수 있겠어요!")
    
    print("\n[✨ Prompt Engineering 예시 ✨]")
    print("=============================================")
    
    # 챗봇 역할을 부여하는 프롬프트 구조를 보여줍니다.
    print("👤 사용자 시스템 프롬프트 (System Prompt):")
    print("당신은 친절하고 유머 감각이 넘치는 'GPT 튜터봇'입니다. 사용자가 질문하면, 명확하게 개념을 설명하고, 관련 예시를 곁들여 쉽게 답변하세요.")
    
    print("\n🧠 사용자 질문 (User Query - 예시):")
    print(f"'{best_topic}' 개념에 대해 초등학교 6학년도 이해할 수 있게 설명해줄 수 있니?")
    
    print("\n💡 기대하는 AI 답변 (Desired AI Response):")
    print("매우 쉽고, 비유를 사용하여 설명해야 합니다. 답변의 끝에는 흥미로운 Q&A 코너를 추가해주세요.")

else:
    print("😅 분석할 만한 충분한 데이터가 나오지 않았습니다. 다음에 더 많은 샘플로 도전해봐요!")

print("\n\n🎉 코딩 탐험 완료! 데이터를 분석하는 능력은 AI 개발의 기본 중 기본이랍니다. 정말 잘했어요! 🎉")